In [1]:
import os
os.chdir('../')

In [2]:
import os, torch
from tqdm import tqdm

def get_clip_score(pt_dir, clip, batch_size=64, device=None):
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    files = sorted(p for p in (os.path.join(pt_dir, f) for f in os.listdir(pt_dir)) if p.endswith(".pt"))
    assert files, f"No .pt files in {pt_dir}"
    use_amp, outs = (dev == "cuda"), []

    with torch.no_grad():
        for i in tqdm(range(0, len(files), batch_size)):
            batch = [torch.load(p, map_location="cpu") for p in files[i:i+batch_size]]
            raws  = [ (d["raw"].unsqueeze(0) if d["raw"].ndim == 3 else d["raw"]) for d in batch ]
            conds = [ d["cond"] for d in batch ]
            raw   = torch.cat(raws, 0).to(dev, dtype=torch.bfloat16)

            with torch.autocast("cuda", torch.bfloat16, enabled=True):
                # reduction='none' → (B,)
                score = 1 - clip.get_cossim_loss(raw, conds, clamp_mode="hard", reduction="none")
            outs.append(score.detach().cpu())

    return float(torch.cat(outs).mean().item())


In [5]:
from utils.clip import CLIPEmbedder
device = 'cuda:0'
model_names = ['ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']

for model_name in model_names:
    clip = CLIPEmbedder(model_name=model_name, device=device)

    for step in [4, 3]:
        pt_dir = f'samplings/SANA/4.5/{step}/Euler/1000/euler_raw/euler_raw_0'
        try:
            score = get_clip_score(pt_dir, clip, batch_size=16, device=device)
            print(model_name, 'NFE :', step, 'Score :', score)
        except FileNotFoundError or AssertionError:
            continue
    print('======')

print('done')

100%|██████████| 63/63 [00:02<00:00, 24.24it/s]


ViT-B/32 NFE : 4 Score : 0.31061917543411255


100%|██████████| 63/63 [00:02<00:00, 23.58it/s]


ViT-B/32 NFE : 3 Score : 0.29765355587005615


100%|██████████| 63/63 [00:02<00:00, 24.92it/s]


...

ViT-L/14 NFE : 3 Score : 0.24471698701381683


100%|██████████| 63/63 [00:07<00:00,  8.81it/s]


ViT-L/14@336px NFE : 4 Score : 0.2658248245716095


100%|██████████| 63/63 [00:07<00:00,  8.24it/s]

ViT-L/14@336px NFE : 3 Score : 0.25097206234931946
done
